# Archived team notebook
See [workflow and limitations](README.md) before running. Outputs and session metadata have been cleared. Supply your own data paths and checkpoints. These notebooks were not rerun during packaging.

# LoRA 訓練資料自動產生器

- GOOGLE掛載多張圖片
- 使用 Meta 的 Segment Anything Model（SAM）自動圈出瑕疵區塊
- 將瑕疵遮罩自動裁圖（固定為 512x512）
- 每張圖AI 配對 prompt 並打包成 LoRA 訓練格式 zip

In [ ]:
# 安裝必要套件
!pip install -q opencv-python matplotlib supervision
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q git+https://github.com/huggingface/huggingface_hub.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

# SAM 模型的儲存路徑（請改成你自己 Google Drive 上的路徑）
checkpoint_path = "/content/drive/MyDrive/lora_dataset/sam_vit_h_4b8939.pth"

# 載入模型與建立預測器
sam = sam_model_registry["vit_h"](checkpoint=checkpoint_path)
predictor = SamPredictor(sam)

print("SAM 模型已載入")

In [ ]:
!ls /content/drive/MyDrive/lora_dataset/dirt_dataset1 | wc -l #圖片數量
!ls /content/drive/MyDrive/lora_dataset/dirt_dataset1 ###

In [ ]:
# 上傳圖片
import os

# 設定你圖片的實際路徑
input_dir = "/content/drive/MyDrive/lora_dataset/dirt_dataset1" ###

# 取得所有 .jpg 或 .JPG 檔（不含子資料夾）
image_files = sorted([
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.lower().endswith(".jpg")
])

In [ ]:
import os
import cv2
import numpy as np
from segment_anything import SamPredictor, sam_model_registry

# 設定輸出路徑（這一段是你要加的）
output_dir = "/content/lora_output/data"
os.makedirs(output_dir, exist_ok=True)

# 掃描圖片
image_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.lower().endswith(".jpg")]

# 處理圖片
for idx, fname in enumerate(image_files, start=1):
    print(f"\n 處理第 {idx}/{len(image_files)} 張圖片：{fname}")
    img = cv2.imread(fname)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img.shape[:2]
    predictor.set_image(rgb)

    input_point = np.array([[width // 2, height // 2]])
    input_label = np.array([1])

    masks, scores, _ = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True
    )

    for i, mask in enumerate(masks):
        ys, xs = np.where(mask)
        if len(xs) == 0 or len(ys) == 0:
            continue

        x_min, x_max = max(min(xs)-10, 0), min(max(xs)+10, width)
        y_min, y_max = max(min(ys)-10, 0), min(max(ys)+10, height)

        data = img[y_min:y_max, x_min:x_max]
        if data.shape[0] < 64 or data.shape[1] < 64:
            continue

        data = cv2.resize(data, (512, 512))

        # 儲存裁圖
        base = os.path.splitext(os.path.basename(fname))[0]
        save_name = f"{base}_data{i+1}.jpg"
        cv2.imwrite(os.path.join(output_dir, save_name), data)
        print(f"✅ 已儲存：{save_name}")

        if i == 1:  # 最多取前 2 個遮罩
            break

In [ ]:
!cp -r /content/lora_output/data /content/drive/MyDrive/lora_dataset/data